# 04 — Support Ticket Triage, PII Redaction & Guardrails

**Part C**

This notebook demonstrates the project's ticket triage pipeline: inspect support tickets, redact personal information, classify issues into allowed categories, handle malformed responses, compare zero-shot/few-shot prompts and demonstrate token-budget routing.

In [1]:
from pathlib import Path
import os, sys

# Find the project root whether this notebook is opened from the project folder or notebooks/.
CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Instamart project folder (or its notebooks folder).")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: d:\New folder\Instamart capstone project


In [2]:
import pandas as pd
from src.ticket_triage import (
    redact_pii, parse_model_response, run_triage_pipeline, evaluate_and_route,
    ZERO_SHOT_SYSTEM_PROMPT, FEW_SHOT_SYSTEM_PROMPT
)

## 1. Load the support ticket dataset

In [3]:
tickets = pd.read_csv(PROJECT_ROOT / "data/raw/tickets.csv", keep_default_na=False)
print("Ticket dataset:", tickets.shape)
display(tickets.head())
display(tickets["true_category"].value_counts().to_frame("ticket_count"))

Ticket dataset: (164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical
3,T0159,C1172,,,none
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud


,ticket_count
true_category,
delivery,43
fraud,39
billing,39
technical,39
none,4


## 2. PII redaction demonstration
The project uses regular expressions to replace email addresses, Indian mobile-number patterns and order IDs with placeholders. The function returns both the redacted text and counts for each type.

In [4]:
example_text = "Please contact me at customer@example.com or 9876543210 about ORD-123456."
redacted_text, email_count, phone_count, order_count = redact_pii(example_text)
print("Original:", example_text)
print("Redacted:", redacted_text)
print({"emails": email_count, "phones": phone_count, "order_ids": order_count})

Original: Please contact me at customer@example.com or 9876543210 about ORD-123456.
Redacted: Please contact me at [EMAIL] or [PHONE] about [ORDER_ID].
{'emails': 1, 'phones': 1, 'order_ids': 1}


## 3. Validate model response parsing
The parser first tries valid JSON with an allowed category. If that fails, it attempts a single category keyword match. Otherwise it flags the output as invalid.

In [5]:
examples = [
    '{"category": "delivery", "reason": "late order"}',
    'This is a billing issue',
    '{"category": "weather", "reason": "not supported"}'
]
for response in examples:
    print(response, "->", parse_model_response(response))

{"category": "delivery", "reason": "late order"} -> ('classified_json', 'delivery')
This is a billing issue -> ('classified_keyword', 'billing')
{"category": "weather", "reason": "not supported"} -> ('flagged_invalid_output', None)


## 4. Run zero-shot and few-shot triage
The model in this project is a deterministic mock LLM, not a call to a live hosted language model. This makes the demonstration repeatable and avoids external API credentials.

In [6]:
zero_results, zero_tokens, zero_pii = run_triage_pipeline(ZERO_SHOT_SYSTEM_PROMPT, token_budget=None, is_few_shot=False)
print("Zero-shot tokens:", zero_tokens, "PII counts:", zero_pii)
display(zero_results.head())
print("Zero-shot status:")
display(zero_results["status"].value_counts().to_frame("tickets"))

[RETRY] ticket_id=T0002, retry=0, wait=0.0540s
[RETRY] ticket_id=T0116, retry=0, wait=0.0533s
[RETRY] ticket_id=T0153, retry=0, wait=0.0494s
[RETRY] ticket_id=T0039, retry=0, wait=0.0567s
[RETRY] ticket_id=T0152, retry=0, wait=0.0571s
[RETRY] ticket_id=T0043, retry=0, wait=0.0517s
[RETRY] ticket_id=T0111, retry=0, wait=0.0477s
[RETRY] ticket_id=T0155, retry=0, wait=0.0504s
[RETRY] ticket_id=T0100, retry=0, wait=0.0540s
[RETRY] ticket_id=T0068, retry=0, wait=0.0569s
[RETRY] ticket_id=T0135, retry=0, wait=0.0573s
[RETRY] ticket_id=T0009, retry=0, wait=0.0513s
[RETRY] ticket_id=T0067, retry=0, wait=0.0572s
[RETRY] ticket_id=T0106, retry=0, wait=0.0518s
[RETRY] ticket_id=T0097, retry=0, wait=0.0508s
[RETRY] ticket_id=T0003, retry=0, wait=0.0457s
[RETRY] ticket_id=T0019, retry=0, wait=0.0451s
[RETRY] ticket_id=T0086, retry=0, wait=0.0512s
[RETRY] ticket_id=T0065, retry=0, wait=0.0545s
Zero-shot tokens: 12364 PII counts: {'email': 55, 'phone': 91, 'order_id': 132}


,ticket_id,customer_id,status,category,true_category,tokens_used
0,T0024,C0486,classified_json,fraud,fraud,83
1,T0073,C0080,classified_json,billing,billing,79
2,T0095,C1087,classified_json,billing,technical,81
3,T0159,C1172,skipped_empty,,none,0
4,T0080,C0910,classified_json,billing,fraud,83


Zero-shot status:


,tickets
status,
classified_json,135
classified_keyword,12
flagged_invalid_output,9
skipped_empty,4
flagged_long,4


In [7]:
few_results, few_tokens, few_pii = run_triage_pipeline(FEW_SHOT_SYSTEM_PROMPT, token_budget=None, is_few_shot=True)
print("Few-shot tokens:", few_tokens, "PII counts:", few_pii)
display(few_results.head())
classified = few_results[few_results["status"].isin(["classified_json", "classified_keyword"])]
accuracy = (classified["category"] == classified["true_category"]).mean()
print(f"Few-shot accuracy among classified tickets: {accuracy:.1%}")

[RETRY] ticket_id=T0002, retry=0, wait=0.0502s


[RETRY] ticket_id=T0116, retry=0, wait=0.0474s
[RETRY] ticket_id=T0153, retry=0, wait=0.0533s
[RETRY] ticket_id=T0039, retry=0, wait=0.0464s
[RETRY] ticket_id=T0152, retry=0, wait=0.0491s
[RETRY] ticket_id=T0043, retry=0, wait=0.0469s
[RETRY] ticket_id=T0111, retry=0, wait=0.0505s
[RETRY] ticket_id=T0155, retry=0, wait=0.0470s
[RETRY] ticket_id=T0100, retry=0, wait=0.0471s
[RETRY] ticket_id=T0068, retry=0, wait=0.0546s
[RETRY] ticket_id=T0135, retry=0, wait=0.0473s
[RETRY] ticket_id=T0009, retry=0, wait=0.0536s
[RETRY] ticket_id=T0067, retry=0, wait=0.0505s
[RETRY] ticket_id=T0106, retry=0, wait=0.0454s
[RETRY] ticket_id=T0097, retry=0, wait=0.0508s
[RETRY] ticket_id=T0003, retry=0, wait=0.0501s
[RETRY] ticket_id=T0019, retry=0, wait=0.0537s
[RETRY] ticket_id=T0086, retry=0, wait=0.0480s
[RETRY] ticket_id=T0065, retry=0, wait=0.0497s
Few-shot tokens: 20632 PII counts: {'email': 55, 'phone': 91, 'order_id': 132}


,ticket_id,customer_id,status,category,true_category,tokens_used
0,T0024,C0486,classified_json,fraud,fraud,136
1,T0073,C0080,classified_json,billing,billing,132
2,T0095,C1087,classified_json,technical,technical,134
3,T0159,C1172,skipped_empty,,none,0
4,T0080,C0910,classified_json,fraud,fraud,136


Few-shot accuracy among classified tickets: 100.0%


## 5. Compare classification outcomes

In [8]:
def triage_metrics(frame, label):
    valid = frame[frame["status"].isin(["classified_json", "classified_keyword"])]
    return {"prompt": label, "classified": len(valid), "flagged": len(frame)-len(valid),
            "accuracy_on_classified": (valid["category"] == valid["true_category"]).mean(),
            "tokens": zero_tokens if label == "zero-shot" else few_tokens}
comparison = pd.DataFrame([triage_metrics(zero_results, "zero-shot"), triage_metrics(few_results, "few-shot")])
display(comparison)

,prompt,classified,flagged,accuracy_on_classified,tokens
0,zero-shot,147,17,0.857143,12364
1,few-shot,147,17,1.000000,20632


## 6. Run the complete evaluation and token-budget routing checks
This calls the project's full evaluation function, including PII audit, both prompt styles, token budget demonstration and routing checks.

In [9]:
evaluate_and_route()

================ PART C: SUPPORT TICKET TRIAGE ================

--- 1. PII Redaction & Input Guardrail Verification ---
Empty tickets: 4 (Reference: 4)
Long tickets (>400 tokens): 4 (Reference: 4)

--- 2. Running Zero-Shot Pipeline (No Budget) ---
[RETRY] ticket_id=T0002, retry=0, wait=0.0504s
[RETRY] ticket_id=T0116, retry=0, wait=0.0516s
[RETRY] ticket_id=T0153, retry=0, wait=0.0495s
[RETRY] ticket_id=T0039, retry=0, wait=0.0549s
[RETRY] ticket_id=T0152, retry=0, wait=0.0494s
[RETRY] ticket_id=T0043, retry=0, wait=0.0549s
[RETRY] ticket_id=T0111, retry=0, wait=0.0513s
[RETRY] ticket_id=T0155, retry=0, wait=0.0478s
[RETRY] ticket_id=T0100, retry=0, wait=0.0457s
[RETRY] ticket_id=T0068, retry=0, wait=0.0495s
[RETRY] ticket_id=T0135, retry=0, wait=0.0519s
[RETRY] ticket_id=T0009, retry=0, wait=0.0571s
[RETRY] ticket_id=T0067, retry=0, wait=0.0467s
[RETRY] ticket_id=T0106, retry=0, wait=0.0509s
[RETRY] ticket_id=T0097, retry=0, wait=0.0516s
[RETRY] ticket_id=T0003, retry=0, wait=0.0485s

{'df_zero':     ticket_id customer_id                  status  category true_category  \
 0       T0024       C0486         classified_json     fraud         fraud   
 1       T0073       C0080         classified_json   billing       billing   
 2       T0095       C1087         classified_json   billing     technical   
 3       T0159       C1172           skipped_empty                    none   
 4       T0080       C0910         classified_json   billing         fraud   
 ..        ...         ...                     ...       ...           ...   
 159     T0150       C0622         classified_json  delivery      delivery   
 160     T0065       C0205  flagged_invalid_output                 billing   
 161     T0128       C0467         classified_json     fraud         fraud   
 162     T0005       C0404         classified_json   billing       billing   
 163     T0098       C0343      classified_keyword  delivery      delivery   
 
      tokens_used  
 0             83  
 1         

## 7. Interpretation
1. Why should PII be redacted before sending ticket text to a model?
2. What happens when the model returns malformed or unsupported JSON?
3. What trade-off can few-shot examples introduce in token usage?
4. Why should invalid/uncertain tickets be flagged or routed for human review?

**Part C complete:** ticket triage, PII redaction, response validation and budget guardrails have been demonstrated.